In [ ]:
# 1. Set up MsipNet conda environment
!git clone https://github.com/NJAU-CDSIC/MsipNet.git
%cd MsipNet

!conda env create -f environment.yml


In [ ]:

# 2. Set up RNA-FM

%cd ~
!git clone https://github.com/ml4bio/RNA-FM.git
%cd RNA-FM
!conda env create -f environment.yml

# install after fixing broken setup.py referencing non existent md file
!pip install .


In [ ]:


# 3. Download pretrained RNA-FM weights

%cd ~/RNA-FM/redevelop/pretrained/
!wget https://huggingface.co/cuhkaih/rnafm/resolve/main/RNA-FM_pretrained.pth

# Fix 403 error on auto-download: manually place weights where torch.hub expects them
!mkdir -p ~/.cache/torch/hub/checkpoints/
!cp ~/RNA-FM/redevelop/pretrained/RNA-FM_pretrained.pth ~/.cache/torch/hub/checkpoints/
# Verify RNA-FM loads correctly
import fm
model, alphabet = fm.pretrained.rna_fm_t12()


In [ ]:

# 4. Prepare FASTA files from parquet (positives + negatives)

%cd ~/clip-gnn-pipeline-main/ssd1_custom_intervals_vienna/
!python parquet2Fasta.py

#construct TSV needed to fit MSIPNET
!python parquet2TSV.py
# produces: Ssd1peaks_norm100_no_overlaps.fasta (positives)
#           Ssd1peaks_norm100_negatives.fasta (negatives)



In [ ]:

# 5. Generate RNA-FM embeddings: positives

%cd ~/RNA-FM/redevelop
!python launch/predict.py \
    --config="pretrained/extract_embedding.yml" \
    --data_path="/home/s2850039/clip-gnn-pipeline-main/msipnet/Ssd1peaks_norm100_no_overlaps.fasta" \
    --save_dir="/home/s2850039/clip-gnn-pipeline-main/msipnet/results_pos" \
    --save_frequency 1 \
    --save_embeddings



In [ ]:

# 6. Generate RNA-FM embeddings: negatives

!python launch/predict.py \
    --config="pretrained/extract_embedding.yml" \
    --data_path="/home/s2850039/clip-gnn-pipeline-main/msipnet/Ssd1peaks_norm100_negatives.fasta" \
    --save_dir="/home/s2850039/clip-gnn-pipeline-main/msipnet/results_neg" \
    --save_frequency 1 \
    --save_embeddings



In [ ]:

import glob
import numpy as np

files = glob.glob('results_pos/representations/*.npy')
print(len(files), "files found")
print(files[0])  # see filename

sample = np.load(files[0])
print(sample.shape)

In [ ]:
%cd ~/clip-gnn-pipeline-main/msipnet
import os
import numpy as np
import torch
import pandas as pd

path_pos = 'results_pos/representations'
path_neg = 'results_neg/representations'

records = {}  # header -> (array, label)

for path, label in [(path_pos, 1), (path_neg, 0)]:
    for fname in os.listdir(path):
        header = os.path.splitext(fname)[0]
        data = np.load(os.path.join(path, fname))
        records[header] = (data, label)

df = pd.read_parquet("ssd1_clip.parquet")

ordered_embeddings = []
ordered_labels = []

for header in df["fasta_header"]:
    arr, label = records[header]
    ordered_embeddings.append(torch.from_numpy(arr))
    ordered_labels.append(label)

print(f"Total sequences: {len(records)}")

# Stack into one tensor
embeddings_tensor = torch.stack(ordered_embeddings)
print("Final shape:", embeddings_tensor.shape)

torch.save(embeddings_tensor, "Ssd1.pt")


In [ ]:

# 10. Move files into MsipNet's expected locations

!mkdir -p ~/MsipNet/Datasets/CLIP_seq/
!mkdir -p ~/MsipNet/MsipNet_code/FM_embedding
!cp Ssd1.tsv ~/MsipNet/Datasets/CLIP_seq/
!cp Ssd1.pt ~/MsipNet/MsipNet_code/FM_embedding/



In [ ]:

# 11. Train MsipNet (sequence-only)
%cd ~/MsipNet/Scripts/Sequence_only

!python main_seq.py --data_file Ssd1 --train --seed 42 --early_stopping 5

/home/s2850039/MsipNet/Scripts/Sequence_only
FM_embeddin: torch.Size([28537, 100, 640])
mer: (28537, 64, 98)
Fold 1/5
Ssd1 	 Train Epoch: 1     avg.loss: 0.0227 ACC: 0.52%, PR: 0.5236, Recall: 0.9840, Specificity: 0.0227, MCC: 0.0125 ,F1-socre: 0.6790, AUC: 0.5748, AP: 0.6075 lr: 0.000103
Ssd1 	 Test  Epoch: 1     avg.loss: 0.0459 ACC: 0.54%, PR: 0.5321 , Recall: 0.9760 ,Specificity: 0.0459, MCC: 0.0601, F1-socre: 0.6887, AUC: 0.6038 (0.6038), AP: 0.6243 1
Ssd1 	 Train Epoch: 2     avg.loss: 0.1043 ACC: 0.55%, PR: 0.5402, Recall: 0.9567, Specificity: 0.1043, MCC: 0.0948 ,F1-socre: 0.6848, AUC: 0.6398, AP: 0.6696 lr: 0.000107
Ssd1 	 Test  Epoch: 2     avg.loss: 0.2013 ACC: 0.58%, PR: 0.5611 , Recall: 0.9185 ,Specificity: 0.2013, MCC: 0.1732, F1-socre: 0.6966, AUC: 0.6687 (0.6687), AP: 0.6904 2
Ssd1 	 Train Epoch: 3     avg.loss: 0.1935 ACC: 0.57%, PR: 0.5581, Recall: 0.9252, Specificity: 0.1935, MCC: 0.1682 ,F1-socre: 0.6898, AUC: 0.6812, AP: 0.7040 lr: 0.000111
Ssd1 	 Test  Epoch: 3   

In [ ]:
%cd ../Motif_discovery
!python motif.py --file_name Ssd1